# GEDI L4A pre/post overlap — feasibility assessment (Amazon)

Goal: determine whether co-located, quality-filtered GEDI L4A shot pairs (10 m or 25 m apart) bracketing a Hansen forest-loss event can be discovered efficiently in the Amazon, to support a PALSAR-2 SAR-change vs ΔAGBD calibration figure for a NISAR proposal.

Compares four query paths over a small test AOI:
1. **GEE shot-level** `LARSE/GEDI/GEDI04_A_002` + spatial self-join (recommended)
2. **GEE monthly raster** `LARSE/GEDI/GEDI04_A_002_MONTHLY` co-located valid-pixel mask
3. **earthaccess** granule listing (no HDF5 download)
4. **gedidb** — only attempted if 1–3 fail

Plus a PALSAR-2 ScanSAR + annual mosaic coverage check.

No HDF5 files are downloaded. All numbers are produced server-side or as small JSON responses.

## Setup

In [1]:
from __future__ import annotations

import logging
import time
from contextlib import contextmanager
from dataclasses import dataclass, field

import ee

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
log = logging.getLogger("gedi_overlap")

ee.Initialize(
    project="dyce-biomass",
    opt_url="https://earthengine-highvolume.googleapis.com",
)
log.info("Earth Engine initialized")

2026-04-26 00:40:04,681 INFO Earth Engine initialized


In [2]:
@contextmanager
def timed(label: str):
    t0 = time.perf_counter()
    log.info("START %s", label)
    try:
        yield
    finally:
        dt = time.perf_counter() - t0
        log.info("END   %s (%.2fs)", label, dt)


@dataclass
class MethodResult:
    name: str
    installed: bool = True
    latency_s: float | None = None
    pairs_25m: int | None = None
    pairs_10m: int | None = None
    pairs_25m_disturbed: int | None = None
    supports_quality_flags: bool = False
    supports_polygon_aoi: bool = False
    returns_points: bool = False
    notes: str = ""


results: dict[str, MethodResult] = {}

## Phase 1 — Test AOI and pair criteria

Small AOI in southern Pará, Brazil, on the active deforestation arc near Novo Progresso. Keeping it small (~0.5°×0.5°) to bound GEE join cost; can be enlarged later.

Pair criteria:
- shot A: 2019-04-18 → 2020-12-31
- shot B: 2022-01-01 → 2023-03-16 (end of GEDI Phase 1)
- both: `l4_quality_flag == 1`, `degrade_flag == 0`
- horizontal separation reported at 10 m (strict) and 25 m (footprint) thresholds
- Hansen `lossyear` ∈ {2020, 2021, 2022} at the pair location

In [7]:
# AOI: 0.5° box, southern Pará near Novo Progresso
# AOI_BOUNDS = (-55.5, -7.0, -55.0, -6.5)  # (west, south, east, north) deg
# AOI = ee.Geometry.Rectangle(list(AOI_BOUNDS))

AOIs = {
    1: ee.Geometry.Polygon(
        [[[-66.4724905067881, -8.730958829551433],
          [-66.4724905067881, -9.706944326978995],
          [-65.0552541786631, -9.706944326978995],
          [-65.0552541786631, -8.730958829551433]]]),
    2: ee.Geometry.Polygon(
        [[[-75.1736623817881, -6.509685094439361],
          [-75.1736623817881, -11.585569659656471],
          [-67.6150686317881, -11.585569659656471],
          [-67.6150686317881, -6.509685094439361]]])
}
AOI_BOUNDS = {
    1: (-66.4724905067881, -9.706944326978995, -65.0552541786631, -8.730958829551433),  # (west, south, east, north) deg
    2: (-75.1736623817881, -11.585569659656471, -67.6150686317881, -6.509685094439361)
}
      
AOI = AOIs[1]
AOI_BOUNDS = AOI_BOUNDS[1]

PRE_START, PRE_END = "2019-04-18", "2021-01-01"
POST_START, POST_END = "2022-01-01", "2023-03-17"

DIST_10_M = 10.0
DIST_25_M = 25.0
DIST_MAX_ERROR_M = 1.0  # GEE projection tolerance for withinDistance

HANSEN_LOSS_YEARS = (20, 21, 22)  # encoded years 2020-2022 in Hansen lossyear band

log.info("AOI bounds (W,S,E,N) = %s", AOI_BOUNDS)
log.info("AOI area km^2 ≈ %.0f", AOI.area(maxError=10).divide(1e6).getInfo())

2026-04-26 09:01:48,989 INFO AOI bounds (W,S,E,N) = (-66.4724905067881, -9.706944326978995, -65.0552541786631, -8.730958829551433)
2026-04-26 09:01:49,588 INFO Refreshing credentials due to a 401 response. Attempt 1/2.
2026-04-26 09:01:50,632 INFO AOI area km^2 ≈ 16882


## Phase 2.1 — GEE shot-level path

Filter `LARSE/GEDI/GEDI04_A_002` (each feature is one shot, point geometry) to AOI + dates + quality. Self-join pre × post at 25 m and 10 m.

Then sample Hansen `lossyear` at the pre-shot location and keep only pairs whose pixel encodes a 2020–2022 loss year.

In [8]:
GEDI04A_INDEX = ee.FeatureCollection("LARSE/GEDI/GEDI04_A_002_INDEX")
HANSEN = ee.Image("UMD/hansen/global_forest_change_2024_v1_12")
HANSEN_LOSSYEAR = HANSEN.select("lossyear")
DISTURBED_MASK = HANSEN_LOSSYEAR.gte(20).And(HANSEN_LOSSYEAR.lte(22))


def gedi_table_ids(start_iso: str, end_iso: str) -> list[str]:
    """Get per-granule asset IDs intersecting AOI in [start, end)."""
    sub = (
        GEDI04A_INDEX.filterBounds(AOI)
        .filter(ee.Filter.gte("time_start", start_iso))
        .filter(ee.Filter.lt("time_start", end_iso))
    )
    return sub.aggregate_array("table_id").getInfo()


def gedi_quality_filter(fc: ee.FeatureCollection) -> ee.FeatureCollection:
    return fc.filter(
        ee.Filter.And(
            ee.Filter.eq("l4_quality_flag", 1),
            ee.Filter.eq("degrade_flag", 0),
        )
    )


def disturbance_filter(fc: ee.FeatureCollection) -> ee.FeatureCollection:
    """Keep only shots whose location has Hansen lossyear in 2020\u20132022."""
    return DISTURBED_MASK.rename("dist").sampleRegions(
        collection=fc, scale=30, geometries=True, tileScale=4
    ).filter(ee.Filter.eq("dist", 1))


def build_window_fc(start_iso: str, end_iso: str) -> ee.FeatureCollection:
    ids = gedi_table_ids(start_iso, end_iso)
    log.info("    %d granules in window", len(ids))
    if not ids:
        return ee.FeatureCollection([])
    parts = [
        gedi_quality_filter(ee.FeatureCollection(tid).filterBounds(AOI))
        for tid in ids
    ]
    return ee.FeatureCollection(parts).flatten()


PRE_START_ISO = PRE_START + "T00:00:00Z"
PRE_END_ISO = PRE_END + "T00:00:00Z"
POST_START_ISO = POST_START + "T00:00:00Z"
POST_END_ISO = POST_END + "T00:00:00Z"

with timed("build pre-window FC (quality-filtered)"):
    pre_full = build_window_fc(PRE_START_ISO, PRE_END_ISO)
with timed("build post-window FC (quality-filtered)"):
    post_full = build_window_fc(POST_START_ISO, POST_END_ISO)

# Pre-filter by Hansen disturbance mask BEFORE the join \u2014 dramatically shrinks it.
with timed("Hansen-disturbed pre-shots"):
    pre = disturbance_filter(pre_full)
    n_pre_disturbed = pre.size().getInfo()
with timed("Hansen-disturbed post-shots"):
    post = disturbance_filter(post_full)
    n_post_disturbed = post.size().getInfo()

log.info("disturbed pre n=%d, disturbed post n=%d", n_pre_disturbed, n_post_disturbed)


2026-04-26 09:02:01,452 INFO START build pre-window FC (quality-filtered)
2026-04-26 09:02:04,089 INFO     70 granules in window
2026-04-26 09:02:04,109 INFO END   build pre-window FC (quality-filtered) (2.66s)
2026-04-26 09:02:04,111 INFO START build post-window FC (quality-filtered)
2026-04-26 09:02:05,107 INFO     77 granules in window
2026-04-26 09:02:05,120 INFO END   build post-window FC (quality-filtered) (1.01s)
2026-04-26 09:02:05,121 INFO START Hansen-disturbed pre-shots
2026-04-26 09:06:12,161 INFO END   Hansen-disturbed pre-shots (247.04s)
2026-04-26 09:06:12,163 INFO START Hansen-disturbed post-shots
2026-04-26 09:10:05,854 INFO END   Hansen-disturbed post-shots (233.69s)
2026-04-26 09:10:05,857 INFO disturbed pre n=28081, disturbed post n=15836


In [11]:
joined_25.size().getInfo()

EEException: User memory limit exceeded.

In [9]:
def join_within(pre_fc, post_fc, distance_m: float):
    distance_filter = ee.Filter.withinDistance(
        distance=distance_m, leftField=".geo", rightField=".geo", maxError=DIST_MAX_ERROR_M
    )
    join = ee.Join.saveAll(matchesKey="matches", measureKey="distance", outer=False)
    return join.apply(pre_fc, post_fc, distance_filter)


def n_pairs(joined: ee.FeatureCollection) -> int:
    """Sum over pre-shots of #post-matches (each (pre, post) counted once)."""
    counts = joined.aggregate_array("matches").map(lambda x: ee.List(x).size())
    return int(ee.Number(counts.reduce(ee.Reducer.sum())).getInfo() or 0)


shot_result = MethodResult(
    name="GEE shot-level (LARSE/GEDI/GEDI04_A_002 via INDEX)",
    supports_quality_flags=True,
    supports_polygon_aoi=True,
    returns_points=True,
    notes="INDEX \u2192 per-granule FC merge \u2192 disturbance prefilter \u2192 self-join",
)

if n_pre_disturbed == 0 or n_post_disturbed == 0:
    log.warning("No disturbed shots in one of the windows \u2014 skipping join")
    shot_result.pairs_25m = 0
    shot_result.pairs_10m = 0
    shot_result.pairs_25m_disturbed = 0
    shot_result.latency_s = 0.0
else:
    t0 = time.perf_counter()
    with timed("self-join @ 25 m on disturbed shots"):
        joined_25 = join_within(pre, post, DIST_25_M)
        pairs_25 = n_pairs(joined_25)
    with timed("self-join @ 10 m on disturbed shots"):
        joined_10 = join_within(pre, post, DIST_10_M)
        pairs_10 = n_pairs(joined_10)
    shot_result.latency_s = time.perf_counter() - t0
    shot_result.pairs_25m = pairs_25
    shot_result.pairs_25m_disturbed = pairs_25  # all are on disturbed pixels
    shot_result.pairs_10m = pairs_10
    log.info("disturbed pairs @ 25 m=%d   @ 10 m=%d", pairs_25, pairs_10)


2026-04-26 09:27:49,623 INFO START self-join @ 25 m on disturbed shots
2026-04-26 09:28:13,731 INFO END   self-join @ 25 m on disturbed shots (24.11s)


EEException: User memory limit exceeded.

### Disturbance filter via Hansen `lossyear`

For each pre-shot with ≥1 post-match at 25 m, sample Hansen `lossyear` at the pre-shot location. Keep pairs where lossyear ∈ {20, 21, 22} (i.e. between visits).

In [ ]:
# Disturbance filter is applied BEFORE the join (see previous cell), so all pairs are disturbed.
# This cell records the results into the comparison table.
results[shot_result.name] = shot_result
log.info("recorded shot-level result: %s", shot_result)


### Sample inspection

Pull a few disturbed pairs to a local `geopandas` GeoDataFrame for manual inspection (lightweight — limit 10).

In [ ]:
# Lightweight sample export: use aggregate_array on a few props + geometry to avoid GEE memory caps
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point


def sample_pairs(joined: ee.FeatureCollection, n: int = 10):
    """Return up to n (pre, post) pair previews without serializing nested matches."""
    head = joined.limit(n)
    # Flatten matches client-side to avoid deep nested .getInfo
    feats = head.getInfo().get("features", [])
    rows = []
    for f in feats:
        p = f.get("properties", {})
        matches = p.get("matches") or []
        pre_geom = f.get("geometry", {}).get("coordinates")
        for m in matches[:1]:  # take first match per pre-shot
            mp = m.get("properties", {})
            mg = m.get("geometry", {}).get("coordinates")
            rows.append({
                "pre_lon": pre_geom[0] if pre_geom else None,
                "pre_lat": pre_geom[1] if pre_geom else None,
                "post_lon": mg[0] if mg else None,
                "post_lat": mg[1] if mg else None,
                "pre_agbd": p.get("agbd"),
                "post_agbd": mp.get("agbd"),
                "pre_sensitivity": p.get("sensitivity"),
                "post_sensitivity": mp.get("sensitivity"),
                "pre_beam": p.get("beam"),
                "post_beam": mp.get("beam"),
            })
    return pd.DataFrame(rows)


if shot_result.pairs_25m and shot_result.pairs_25m > 0:
    try:
        sample_df = sample_pairs(joined_25, n=10)
        print(sample_df)
        sample_df.to_csv("sample_pairs.csv", index=False)
        log.info("wrote sample_pairs.csv with %d rows", len(sample_df))
    except Exception as exc:
        log.warning("sample export failed (likely GEE memory cap): %s", exc)
else:
    log.warning("No disturbed pre/post pairs at 25 m \u2014 try a larger AOI or longer windows")


## Phase 2.2 — GEE monthly raster path

Use `LARSE/GEDI/GEDI04_A_002_MONTHLY` (25 m, monthly composites of orbits). For each pre/post window, build a `agbd`-valid pixel count image, then count pixels that have valid coverage in BOTH windows. Pixel co-location is implicit at the 25 m pixel grid.

In [ ]:
GEDI04A_MONTHLY = ee.ImageCollection("LARSE/GEDI/GEDI04_A_002_MONTHLY")


def quality_mask(im: ee.Image) -> ee.Image:
    return (
        im.updateMask(im.select("l4_quality_flag").eq(1))
          .updateMask(im.select("degrade_flag").eq(0))
    )


def valid_pixel_image(start: str, end: str) -> ee.Image:
    coll = (
        GEDI04A_MONTHLY.filterDate(start, end)
        .filterBounds(AOI)
        .map(quality_mask)
        .select("agbd")
    )
    # 1 wherever any month had a valid agbd, masked elsewhere
    return coll.count().gt(0).selfMask()


raster_result = MethodResult(
    name="GEE monthly raster (LARSE/GEDI/GEDI04_A_002_MONTHLY)",
    supports_quality_flags=True,
    supports_polygon_aoi=True,
    returns_points=False,
    notes="co-location implicit at 25 m raster grid; no <25 m sensitivity",
)

t0 = time.perf_counter()
with timed("monthly raster co-located pixel count"):
    pre_valid = valid_pixel_image(PRE_START, PRE_END)
    post_valid = valid_pixel_image(POST_START, POST_END)
    both = pre_valid.And(post_valid)
    pix_count = (
        both.rename("both")
        .reduceRegion(
            reducer=ee.Reducer.sum(),
            geometry=AOI,
            scale=25,
            maxPixels=1e10,
            tileScale=4,
        )
        .get("both")
        .getInfo()
    )
raster_result.latency_s = time.perf_counter() - t0
raster_result.pairs_25m = int(pix_count or 0)
log.info("co-located valid pixels (pre AND post) @ 25 m grid: %s", pix_count)

with timed("monthly raster co-located AND Hansen-disturbed pixel count"):
    loss = HANSEN.select("lossyear")
    disturbed_mask = (
        loss.eq(20).Or(loss.eq(21)).Or(loss.eq(22))
    )
    both_disturbed = both.And(disturbed_mask)
    pix_dist = (
        both_disturbed.rename("both_dist")
        .reduceRegion(
            reducer=ee.Reducer.sum(),
            geometry=AOI,
            scale=25,
            maxPixels=1e10,
            tileScale=4,
        )
        .get("both_dist")
        .getInfo()
    )
raster_result.pairs_25m_disturbed = int(pix_dist or 0)
log.info("co-located AND Hansen-disturbed pixels: %s", pix_dist)

results[raster_result.name] = raster_result

## Phase 2.3 — earthaccess granule listing (no download)

Reuse `nice_sar.search.earthdata.search_earthdata`. Just count granules intersecting the AOI in pre and post windows. No HDF5 streaming.

In [ ]:
from nice_sar.auth.earthdata import login as ed_login  # noqa: E402
from nice_sar.search.earthdata import search_earthdata  # noqa: E402

ea_result = MethodResult(
    name="earthaccess (CMR granule listing)",
    supports_quality_flags=False,  # not at granule level; would require streaming HDF5
    supports_polygon_aoi=True,
    returns_points=False,
    notes="granule-level only; per-shot quality requires streaming HDF5",
)

ea_works = False
try:
    ed_login()
    t0 = time.perf_counter()
    with timed("earthaccess granule listing pre+post"):
        pre_granules = search_earthdata(
            short_name="GEDI_L4A_AGB_Density_V2_1_2056",
            bbox=AOI_BOUNDS,
            temporal=(PRE_START, PRE_END),
    count=2000,
        )
        post_granules = search_earthdata(
            short_name="GEDI_L4A_AGB_Density_V2_1_2056",
            bbox=AOI_BOUNDS,
            temporal=(POST_START, POST_END),
    count=2000,
        )
    ea_result.latency_s = time.perf_counter() - t0
    ea_result.notes += f"; pre granules={len(pre_granules)}, post granules={len(post_granules)}"
    log.info("earthaccess granules: pre=%d post=%d", len(pre_granules), len(post_granules))
    ea_works = True
except Exception as exc:
    log.warning("earthaccess path failed: %s", exc)
    ea_result.installed = False
    ea_result.notes += f"; failed: {exc}"

results[ea_result.name] = ea_result

## Phase 2.4 — gedidb fallback

Only attempted if the GEE shot-level path failed to deliver a workable answer (`pairs_25m_disturbed == 0` or `pairs_25m is None`). Otherwise we record it as not attempted.

In [ ]:
gedidb_result = MethodResult(
    name="gedidb (fallback)",
    installed=False,
    supports_quality_flags=True,
    supports_polygon_aoi=True,
    returns_points=True,
    notes="not attempted: shot-level path succeeded",
)

shot_ok = (
    shot_result.pairs_25m_disturbed is not None and shot_result.pairs_25m_disturbed > 0
)
if not shot_ok:
    log.warning("shot-level path did not deliver disturbed pairs — would attempt gedidb here")
    gedidb_result.notes = "would attempt gedidb in isolated venv; skipped in this run"
results[gedidb_result.name] = gedidb_result

## Phase 3 — PALSAR-2 SAR coverage check

Confirm the SAR side of the proposed figure is feasible over the same AOI/windows. Check ScanSAR (primary, dense, complex) and the annual mosaic (fallback).

In [ ]:
SCANSAR = ee.ImageCollection("JAXA/ALOS/PALSAR-2/Level2_2/ScanSAR")
ANNUAL = ee.ImageCollection("JAXA/ALOS/PALSAR/YEARLY/SAR")


def coverage(coll: ee.ImageCollection, start: str, end: str) -> int:
    return int(
        coll.filterBounds(AOI).filterDate(start, end).size().getInfo() or 0
    )


with timed("PALSAR-2 ScanSAR + annual coverage counts"):
    n_scansar_pre = coverage(SCANSAR, PRE_START, PRE_END)
    n_scansar_post = coverage(SCANSAR, POST_START, POST_END)
    n_annual_pre = coverage(ANNUAL, PRE_START, PRE_END)
    n_annual_post = coverage(ANNUAL, POST_START, POST_END)

log.info(
    "ScanSAR  pre=%d post=%d   |   annual mosaic pre=%d post=%d",
    n_scansar_pre,
    n_scansar_post,
    n_annual_pre,
    n_annual_post,
)

# capture sample ScanSAR metadata so we know what stratification will be needed
scansar_meta_keys = ["system:time_start", "BeamID", "AntennaPointing", "IncAngleNearRange"]
scansar_pre_meta = (
    SCANSAR.filterBounds(AOI)
    .filterDate(PRE_START, PRE_END)
    .limit(5)
    .reduceColumns(ee.Reducer.toList(len(scansar_meta_keys)), scansar_meta_keys)
    .get("list")
    .getInfo()
)
log.info("sample ScanSAR pre metadata (up to 5): %s", scansar_pre_meta)

sar_coverage = {
    "scansar_pre": n_scansar_pre,
    "scansar_post": n_scansar_post,
    "annual_pre": n_annual_pre,
    "annual_post": n_annual_post,
    "scansar_sample_meta": scansar_pre_meta,
}

## Phase 4 — Results table

Tabulate all method results for the report.

In [ ]:
rows = []
for r in results.values():
    rows.append(
        {
            "method": r.name,
            "installed": r.installed,
            "latency_s": None if r.latency_s is None else round(r.latency_s, 1),
            "pairs_10m": r.pairs_10m,
            "pairs_25m": r.pairs_25m,
            "pairs_25m_disturbed": r.pairs_25m_disturbed,
            "quality_flags": r.supports_quality_flags,
            "polygon_aoi": r.supports_polygon_aoi,
            "returns_points": r.returns_points,
            "notes": r.notes,
        }
    )

summary = pd.DataFrame(rows)
display(summary)

summary_path = "results_summary.csv"
summary.to_csv(summary_path, index=False)
log.info("wrote %s", summary_path)

# also dump SAR coverage to a small JSON
import json  # noqa: E402
with open("sar_coverage.json", "w") as f:
    json.dump(sar_coverage, f, indent=2, default=str)
log.info("wrote sar_coverage.json")

## Notes on cost / scaling

- The shot-level self-join is the primary feasibility test. For larger AOIs (e.g. all of Brazilian Amazon), tile the AOI and aggregate; do not call a single `withinDistance` over millions of features.
- The monthly raster path provides a quick upper-bound check on co-location density but loses sub-pixel sensitivity (no 10 m vs 25 m distinction).
- The earthaccess path is necessary if any per-shot variable absent from the GEE asset is required (full L4A schema), or for off-GEE compute (CHPC, Daskhub).
- The `gedidb` path is fallback only.

Findings flow into [report.md](report.md).